# Integers as sets

A prerequisite for sections 6 and 7 of *The book made faster*.  `BitmapBook` and
`TickArrayBook` keep the occupancy of a side of the book in one integer, and read the best
price and the gaps between occupied levels off it with a handful of operations.  They are the
content of `unito26.lob.binary_gaps`, and this notebook takes them one at a time:

1. `count_trailing_zeros` and `discard_trailing_zeros` — the lowest set bit;
2. `keep_lowest_set_bits` and `keep_highest_set_bits` — cutting a window of $d$ occupied levels;
3. `measure_largest_binary_gap` and `count_binary_gaps` — reading the window.

Each is checked against a reference that works on the binary string, which is slow and
obviously right.  The proofs are in `documentation/integers-in-binary.md`; here the identities
are run, printed and tested.

In [ ]:
import random

from unito26.lob.binary_gaps import (
    count_binary_gaps,
    count_trailing_zeros,
    discard_trailing_zeros,
    keep_highest_set_bits,
    keep_lowest_set_bits,
    measure_largest_binary_gap,
)
from unito26.lob.messages import BUY, SELL, ReportedDepth
from unito26.lob.orderbook import TickArrayBook

## 1. A number as a set

Bit $k$ of $n$ is set exactly when $k$ belongs to the set.  With a floor $p_0$ for the price of
bit 0, position $k = p - p_0$ is a price, and one integer is the occupancy of a side.

`show(n, width)` prints the low `width` bits of `n`, highest on the left.  It reduces `n`
modulo $2^{\text{width}}$ first, which for a negative `n` is what the machine holds in a
register of that width — section 4 relies on this.

In [ ]:
def show(n: int, width: int) -> str:
    return format(n % (1 << width), f"0{width}b")


occupancy = 0
for k in (1, 4, 5, 8):
    occupancy |= 1 << k
print("occupy 1, 4, 5, 8 :", show(occupancy, 10))

occupancy &= ~(1 << 4)
print("vacate 4          :", show(occupancy, 10))

print("is 5 occupied?    :", bool(occupancy & 1 << 5))
print("occupied levels   :", occupancy.bit_count())

A *gap* is a run of zeros lying strictly between two ones.  Trailing zeros lie below
every one, so they are never a gap: `0b100` has none.  Leading zeros are not stored at all.

## 2. Worked examples

Each entry is worked out by hand from the binary digits.  The last column is the largest gap
of the odd part of `n`, for the reason given in section 8.

In [ ]:
WORKED_EXAMPLES = [
    dict(n=0b11010, count_trailing_zeros=1, discard_trailing_zeros=0b1101,
         count_binary_gaps=1, largest_binary_gap=1),
    dict(n=0b100, count_trailing_zeros=2, discard_trailing_zeros=0b1,
         count_binary_gaps=0, largest_binary_gap=0),
    dict(n=0b1001, count_trailing_zeros=0, discard_trailing_zeros=0b1001,
         count_binary_gaps=1, largest_binary_gap=2),
    dict(n=0b101101000, count_trailing_zeros=3, discard_trailing_zeros=0b101101,
         count_binary_gaps=2, largest_binary_gap=1),
    dict(n=0b1000100101, count_trailing_zeros=0, discard_trailing_zeros=0b1000100101,
         count_binary_gaps=3, largest_binary_gap=3),
    dict(n=0b1000000, count_trailing_zeros=6, discard_trailing_zeros=0b1,
         count_binary_gaps=0, largest_binary_gap=0),
    dict(n=0b111, count_trailing_zeros=0, discard_trailing_zeros=0b111,
         count_binary_gaps=0, largest_binary_gap=0),
    dict(n=0, count_trailing_zeros=None, discard_trailing_zeros=0,
         count_binary_gaps=0, largest_binary_gap=0),
]

In [ ]:
def computed(n: int) -> dict:
    return dict(
        n=n,
        count_trailing_zeros=count_trailing_zeros(n) if n else None,
        discard_trailing_zeros=discard_trailing_zeros(n),
        count_binary_gaps=count_binary_gaps(n),
        largest_binary_gap=measure_largest_binary_gap(discard_trailing_zeros(n)),
    )


print(f"{'n':>12}  {'ctz':>4}  {'odd part':>12}  {'gaps':>4}  {'largest':>7}")
for example in WORKED_EXAMPLES:
    assert computed(example["n"]) == example, example
    ctz = example["count_trailing_zeros"]
    print(f"{example['n']:>12b}  {'-' if ctz is None else ctz:>4}  "
          f"{example['discard_trailing_zeros']:>12b}  "
          f"{example['count_binary_gaps']:>4}  {example['largest_binary_gap']:>7}")

## 3. References on the binary string

The tests of `binary_gaps` compare every function with a reference that reads `bin(n)` as
text.  They are reproduced here: each is short enough to be right by inspection, and each
trick below is checked against one of them.

In [ ]:
def runs(n: int) -> list[int]:
    """Lengths of the zero runs lying strictly between ones."""
    return [len(run) for run in bin(n)[2:].strip("0").split("1") if run]


def window(digits: str, count: int) -> str:
    """``digits`` from its first ``1`` to its ``count``-th, or ``"0"`` when it has none."""
    ones = [i for i, digit in enumerate(digits) if digit == "1"][:count]
    return digits[ones[0]:ones[-1] + 1] if ones else "0"


def trailing_zeros(n: int) -> int:
    """Shift right one position at a time until the lowest bit is 1."""
    count = 0
    while n & 1 == 0:
        count += 1
        n >>= 1
    return count


n = 0b101101000
print(bin(n), "runs:", runs(n), " trailing zeros:", trailing_zeros(n))
print("first two ones from the top   :", window(bin(n)[2:], 2))
print("first two ones from the bottom:", window(bin(n)[2:][::-1], 2)[::-1])

`trailing_zeros` takes one step per trailing zero, and does not terminate on `0`.  The
next section does the same in a fixed number of integer operations.

## 4. `count_trailing_zeros`: the lowest set bit

`count_trailing_zeros(n)` is `(n & -n).bit_length() - 1`.  Two facts make it work.

**`bit_length` is a degree.**  For $n > 0$, `n.bit_length()` is one more than the position of
the highest set bit.  So if `n & -n` is a single bit $2^\nu$, its `bit_length() - 1` is $\nu$.

**`n & -n` is the lowest set bit.**  This needs the binary digits of a negative integer.
Python stores integers as sign and magnitude, but the bitwise operators act as though every
integer were written in *two's complement* with infinitely many digits:

- a non-negative integer has its usual digits followed by zeros forever;
- a negative integer $x$ has the complement of the digits of $-1 - x$, so ones forever.

The two are tied by $\sim x = -x - 1$, that is $-x = \sim x + 1$: negate by inverting every
bit and adding one.

In [ ]:
for x in (0, 1, 5, 360, -1, -360):
    assert ~x == -x - 1
    print(f"{x:>5}  ~x = {~x:>5}   low 12 bits of x: {show(x, 12)}")

The infinite tail is not stored anywhere; it is generated.  Right shift floors, so
`x >> k & 1` reads bit $k$ of the infinite expansion, and the shifts of a negative number
settle at $-1$, all of whose bits are one.

In [ ]:
b = 360
print(f"{'k':>2}  {'b >> k':>7} bit   {'-b >> k':>7} bit")
for k in range(12):
    print(f"{k:>2}  {b >> k:>7}  {b >> k & 1}    {-b >> k:>7}  {-b >> k & 1}")

Now write $n = \dots\,x\,1\,0\cdots0$ with $\nu$ trailing zeros.  Inverting gives
$\dots\,\bar x\,0\,1\cdots1$, and adding one carries through the $\nu$ ones and stops at the
zero: $-n = \dots\,\bar x\,1\,0\cdots0$.  So $-n$ agrees with $n$ at and below the lowest set
bit and with $\sim n$ above it, and the AND keeps exactly one bit.

In [ ]:
for name, value in (("b", b), ("~b", ~b), ("-b", -b), ("b & -b", b & -b)):
    print(f"{name:>7}  {show(value, 13)}")

print()
print("(b & -b).bit_length() - 1 =", (b & -b).bit_length() - 1)
print("count_trailing_zeros(b)   =", count_trailing_zeros(b))

Against the one-step-at-a-time loop, on integers far wider than a machine word:

In [ ]:
rng = random.Random(26)
for _ in range(1000):
    n = rng.randint(1, 2**200)
    assert count_trailing_zeros(n) == trailing_zeros(n)
print("count_trailing_zeros agrees with the loop on 1000 integers below 2**200")

Zero has no lowest set bit.  `0 & -0` is `0` and `(0).bit_length() - 1` is $-1$: a
plausible grid position, returned without complaint.  The function therefore refuses zero, and
a book tests for an empty side before it asks for a best price.

In [ ]:
print("(0 & -0).bit_length() - 1 =", (0 & -0).bit_length() - 1)
try:
    count_trailing_zeros(0)
except ValueError as error:
    print("count_trailing_zeros(0)   :", error)

## 5. `discard_trailing_zeros`: the odd part

Shifting right by the number of trailing zeros leaves an odd number with the same ones, in the
same relative positions.  The gaps between ones are unchanged, and the trailing zeros, which
were never a gap, are gone.  Every window cut below is returned in this form.

In [ ]:
for n in (0b11010, 0b101101000, 0b1000000, 0):
    print(f"{n:>10b} -> {discard_trailing_zeros(n):b}")

## 6. `keep_lowest_set_bits`: clearing from the bottom

`m & (m - 1)` clears the lowest set bit of `m`.  Subtracting one turns $\dots x\,1\,0\cdots0$
into $\dots x\,0\,1\cdots1$, and the AND keeps only $x$.  After `count` passes, `rest` holds
every set bit except the lowest `count`, and `n ^ rest` is those `count` bits.

In [ ]:
n, count = 0b1000100101, 3
rest = n
print(f"{'n':<9}{show(n, 10)}")
for step in range(count):
    rest &= rest - 1
    print(f"{f'pass {step + 1}':<9}{show(rest, 10)}")
print(f"{'n ^ rest':<9}{show(n ^ rest, 10)}")
print(f"{'kept':<9}{keep_lowest_set_bits(n, count):b}")

In [ ]:
for count in range(1, 7):
    for n in range(5000):
        expected = int(window(bin(n)[2:][::-1], count)[::-1], 2)
        assert keep_lowest_set_bits(n, count) == expected, (n, count)
print("keep_lowest_set_bits agrees with the string reference")

The loop takes one pass per kept bit, and every pass touches every digit of `n`.

## 7. `keep_highest_set_bits`: bisection on a monotone count

`(n >> k).bit_count()` is the number of set bits at position $k$ and above.  As $k$ rises it
never increases, and it drops by one exactly where bit $k$ is set.

In [ ]:
n = 0b1000100101
print(f"n = {n:b}")
for k in range(n.bit_length() + 1):
    print(f"k = {k:>2}   bit {n >> k & 1}   set bits at or above k: {(n >> k).bit_count()}")

The `count` highest bits start at the largest $k$ where the count still reaches `count`.
A monotone predicate is found by bisection, in $O(\log W)$ population counts for a span of
$W$ bits, whatever `count` is.  The trace below repeats the function's loop with a print.

In [ ]:
def traced_keep_highest_set_bits(n: int, count: int) -> int:
    if n.bit_count() <= count:
        return discard_trailing_zeros(n)
    low, high = 0, n.bit_length() - 1
    while low < high:
        mid = (low + high + 1) // 2
        above = (n >> mid).bit_count()
        print(f"low {low:>2}  high {high:>2}  mid {mid:>2}  bits at or above mid {above}")
        if above >= count:
            low = mid
        else:
            high = mid - 1
    return n >> low


kept = traced_keep_highest_set_bits(0b1000100101, 2)
print(f"kept {kept:b}")
assert kept == keep_highest_set_bits(0b1000100101, 2)

In [ ]:
for count in range(1, 7):
    for n in range(5000):
        expected = int(window(bin(n)[2:], count), 2)
        assert keep_highest_set_bits(n, count) == expected, (n, count)
print("keep_highest_set_bits agrees with the string reference")

Both functions return the same window, shifted so its lowest one is bit 0, and hence odd.
They differ in which end they cut from and in how they find the cut.

## 8. `measure_largest_binary_gap`: flooding

`n |= n >> 1` copies every one a position down, so each gap shrinks by one bit per pass.
`n & (n + 1)` is zero exactly when $n$ is a solid block of ones $0\cdots01\cdots1$, because
adding one then carries into a position where $n$ has a zero.  The number of passes is the
width of the widest gap.

In [ ]:
n = 0b1000100101
passes = 0
print(f"pass 0  {show(n, 10)}   n & (n + 1) = {show(n & (n + 1), 10)}")
while n & (n + 1):
    passes += 1
    n |= n >> 1
    print(f"pass {passes}  {show(n, 10)}   n & (n + 1) = {show(n & (n + 1), 10)}")
print("largest gap:", measure_largest_binary_gap(0b1000100101))

Trailing zeros flood as well, and are then counted as a gap they are not.  This is why
the function asks for an odd argument, and why the windows of sections 6 and 7 are odd.

In [ ]:
n = 0b100
print("runs(0b100)                              :", runs(n))
print("measure_largest_binary_gap(0b100)        :", measure_largest_binary_gap(n))
print("... of discard_trailing_zeros(0b100)     :",
      measure_largest_binary_gap(discard_trailing_zeros(n)))

In [ ]:
for n in range(1, 5000):
    assert measure_largest_binary_gap(discard_trailing_zeros(n)) == max(runs(n), default=0), n
print("measure_largest_binary_gap agrees with the string reference")

## 9. `count_binary_gaps`: one mark per run of ones

Bit $k$ of `n << 1` is bit $k-1$ of `n`.  So bit $k$ of `n & ~(n << 1)` is set exactly when bit
$k$ of `n` is set and bit $k-1$ is not: $k$ is the lowest position of a run of ones.  Between
the lowest and the highest set bit, runs of ones and gaps alternate, beginning and ending with
a run, so there is one gap fewer than there are runs.

In [ ]:
n = 0b1011000110
for name, value in (("n", n), ("n << 1", n << 1), ("~(n << 1)", ~(n << 1)),
                    ("n & ~(n << 1)", n & ~(n << 1))):
    print(f"{name:>14}  {show(value, 12)}")
print()
print("runs of ones:", (n & ~(n << 1)).bit_count(), "  gaps:", count_binary_gaps(n))

In [ ]:
for n in range(1, 5000):
    assert count_binary_gaps(n) == len(runs(n)), n
print("count_binary_gaps agrees with the string reference")

No normalisation is needed here: trailing zeros produce no mark, and the count does not
depend on where the window sits.

## 10. Properties

The checks so far compare a function with a reference on a range of inputs.  A *property* is
a statement that holds for every input without saying what the output is: for example, that
shifting $n$ left by $k$ appends $k$ zeros below it, which moves every gap and creates none, so

$$\texttt{count\_binary\_gaps}(n \ll k) = \texttt{count\_binary\_gaps}(n) \quad \text{for all } n > 0,\ k \ge 0.$$

A property is tested by drawing inputs at random and reporting the first one that fails.

In [ ]:
def check_property(name: str, holds, trials: int = 2000, seed: int = 0) -> None:
    rng = random.Random(seed)
    for _ in range(trials):
        n = rng.randint(1, 2 ** rng.randint(1, 120))
        k = rng.randint(0, 64)
        count = rng.randint(1, 8)
        if not holds(n, k, count):
            raise AssertionError(f"{name}: fails at n = {n:b}, k = {k}, count = {count}")
    print(f"holds on {trials} draws: {name}")


def reverse(n: int) -> int:
    return int(bin(n)[2:][::-1], 2)


def odd(n: int) -> int:
    return discard_trailing_zeros(n)

In [ ]:
check_property(
    "gap count is invariant under a left shift",
    lambda n, k, count: count_binary_gaps(n << k) == count_binary_gaps(n))
check_property(
    "largest gap is invariant under a left shift, once normalised",
    lambda n, k, count: measure_largest_binary_gap(odd(n << k))
    == measure_largest_binary_gap(odd(n)))
check_property(
    "a left shift adds k trailing zeros",
    lambda n, k, count: count_trailing_zeros(n << k) == count_trailing_zeros(n) + k)

Reversing the bit string turns the gaps upside down and keeps them.  This is the property
that lets `BitmapBook` read its bids from the top of the integer and its asks from the bottom
with the same two functions.

In [ ]:
check_property(
    "gap count is invariant under reversal",
    lambda n, k, count: count_binary_gaps(reverse(odd(n))) == count_binary_gaps(n))
check_property(
    "largest gap is invariant under reversal",
    lambda n, k, count: measure_largest_binary_gap(reverse(odd(n)))
    == measure_largest_binary_gap(odd(n)))

Bounds and edge cases say what the output may be without computing it.

In [ ]:
check_property(
    "there are fewer gaps than set bits",
    lambda n, k, count: count_binary_gaps(n) <= n.bit_count() - 1)
check_property(
    "the largest gap is zero exactly when the odd part is a block of ones",
    lambda n, k, count: (measure_largest_binary_gap(odd(n)) == 0)
    == (odd(n) & (odd(n) + 1) == 0))
check_property(
    "the largest gap fits in the span",
    lambda n, k, count: measure_largest_binary_gap(odd(n))
    <= odd(n).bit_length() - n.bit_count())

The two windows keep the right number of bits, and asking for all of them keeps the odd
part.

In [ ]:
check_property(
    "keep_highest_set_bits keeps min(count, bit_count) bits",
    lambda n, k, count: keep_highest_set_bits(n, count).bit_count() == min(count, n.bit_count()))
check_property(
    "keep_lowest_set_bits keeps min(count, bit_count) bits",
    lambda n, k, count: keep_lowest_set_bits(n, count).bit_count() == min(count, n.bit_count()))
check_property(
    "keeping every bit from the top is the odd part",
    lambda n, k, count: keep_highest_set_bits(n, n.bit_count()) == odd(n))
check_property(
    "keeping every bit from the bottom is the odd part",
    lambda n, k, count: keep_lowest_set_bits(n, n.bit_count()) == odd(n))
check_property(
    "the two windows are each other's reversal",
    lambda n, k, count: keep_highest_set_bits(reverse(odd(n)), count)
    == reverse(keep_lowest_set_bits(n, count)))

None of these properties names a value of the function.  Each states something the
function must mean, and a random draw finds counterexamples a hand-picked list would miss.
When a code assistant proposes a faster version, a set of properties like these is a quick
way to check it.

In [ ]:
def flawed_count_binary_gaps(n: int) -> int:
    return bin(n)[2:].count("10")


try:
    check_property(
        "a plausible one-liner agrees with count_binary_gaps",
        lambda n, k, count: flawed_count_binary_gaps(n) == count_binary_gaps(n))
except AssertionError as failure:
    print(failure)

print("on 0b10:", flawed_count_binary_gaps(0b10), "against", count_binary_gaps(0b10))

The one-liner counts the places where a one is followed by a zero, and so counts the
trailing zeros as a gap whenever there are any.  It passes every odd input.

## 11. Back to the book

A `TickArrayBook` counts each side from the band edge behind it: bid at bit `price - floor`,
ask at bit `ceiling - price`.  The touch is the highest set bit on either side, and the levels
beyond the reported depth are the low bits.  The window of the first $d$ reported levels is
therefore `keep_highest_set_bits(bits, d)` on both sides.

In [ ]:
book = TickArrayBook(floor=100, width=20)
for price, size in ((101, 300), (104, 200), (105, 100), (108, 400)):
    book.set_size(BUY, price, size)
for price, size in ((110, 100), (111, 200), (115, 500), (119, 300)):
    book.set_size(SELL, price, size)

depth = ReportedDepth(3)
for direction, name in ((BUY, "bid"), (SELL, "ask")):
    print(f"{name}  occupancy {show(book._bits[direction], book.width)}   "
          f"window {book.span_bits(direction, depth):b}   "
          f"gaps {book.gap_count(direction, depth)}   "
          f"largest {book.largest_gap_size_between_non_empty_levels(direction, depth)}")

The same numbers read off the prices: two consecutive occupied prices bound one run of
empty positions, of length $|p_{i+1} - p_i| - 1$.

In [ ]:
for direction, name in ((BUY, "bid"), (SELL, "ask")):
    prices = sorted(book.levels_map(direction), reverse=direction == BUY)[:depth]
    empty = [abs(q - p) - 1 for p, q in zip(prices, prices[1:]) if abs(q - p) > 1]
    print(f"{name}  reported prices {prices}   gaps {len(empty)}   largest {max(empty, default=0)}")
    assert len(empty) == book.gap_count(direction, depth)
    assert max(empty, default=0) == book.largest_gap_size_between_non_empty_levels(direction, depth)

## 12. What does it print?

Predict each output before running the cell.

In [ ]:
n = 0b10100
print(measure_largest_binary_gap(n))

In [ ]:
n = -12
print(n & -n)

In [ ]:
n = 0b1011000
print(bin(n & ~(n << 1)))

In [ ]:
n = 0b110101
rest = n
rest &= rest - 1
rest &= rest - 1
print(bin(n ^ rest))